# Tutorial 04 - Exercise 04: Hospital Risk Scoring & Pivot Analytics
Computes total costs, patient age segments, composite clinical risk scores, pivot aggregations, and regional exports.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/hospital_original.csv")

# 1. Total Cost & Age Groups
df['TotalCost'] = df['DaysAdmitted'] * df['DailyCost']

def categorize_age(age):
    if age < 30: return "Young"
    elif 30 <= age <= 50: return "Middle"
    else: return "Senior"

df['AgeGroup'] = df['Age'].apply(categorize_age)
df.rename(columns={'DailyCost': 'DailyFee'}, inplace=True)
display(df.head())


### 2. Clinical Risk Score Computation
Formula:
$$\text{RiskScore} = (\text{DaysAdmitted} \times 0.4) + (\text{Age} \times 0.3) + (100 - \text{Satisfaction} \times 10) \times 0.3$$


In [ ]:
df['RiskScore'] = (df['DaysAdmitted'] * 0.4) + (df['Age'] * 0.3) + (100 - df['Satisfaction'] * 10) * 0.3

def categorize_risk(score):
    if score > 60: return "High"
    elif 40 <= score <= 60: return "Medium"
    else: return "Low"

df['RiskCategory'] = df['RiskScore'].apply(categorize_risk)
df.sort_values(by='RiskScore', ascending=False, inplace=True)
display(df[['City', 'Age', 'DaysAdmitted', 'Satisfaction', 'RiskScore', 'RiskCategory']])


### 3. City Summary Aggregations


In [ ]:
city_summary = df.groupby('City').agg(
    AvgTotalCost=('TotalCost', 'mean'),
    AvgSatisfaction=('Satisfaction', 'mean'),
    ReadmissionRate=('Readmitted', 'mean')
).reset_index()

display(city_summary)

highest_risk = df.groupby('City')['RiskScore'].mean().idxmax()
print(f"\nHighest Clinical Risk City: {highest_risk}")


### 4. Data Mapping & Pivot Tables


In [ ]:
df['RegionCode'] = df['City'].map({'Alger': 'DZ-16', 'Annaba': 'DZ-23', 'Oran': 'DZ-31'})
df['ReadmissionStatus'] = df['Readmitted'].map({0: 'No', 1: 'Yes'})

pivot = df.pivot_table(index='City', columns='Department', values=['TotalCost', 'Satisfaction'], aggfunc='mean')
display(pivot)

# Export processed tables
df.to_csv("../data/hospital_risk.csv", index=False)
city_summary.to_csv("../data/city_summary.csv", index=False)
print("Processed tables exported to CSV.")
